## 3 — Analyse Movements: Kinematics and Muscle Synergies

This notebook collects muscle activation data during rollouts and uses Non-Negative Matrix Factorization (NMF) to extract muscle synergies — a key technique in motor control research.

**What you'll learn:**
- How to collect rollout data from any MyoSuite environment
- What muscle synergies are and how to compute them with NMF
- How to plot Variance Accounted For (VAF) curves

**Prerequisites:** Completed notebook 1 · `pip install stable-baselines3 scikit-learn`

> **Note:** 1000 training steps produces a near-random policy. The synergy analysis is meaningful even on random activations — to see task-specific synergies, increase `total_timesteps` to ≥ 200 000 and re-run the analysis cells.

In [ ]:
import gymnasium as gym
import myosuite  # registers env ids
import numpy as np
import os

In [ ]:
from stable_baselines3 import PPO

env = gym.make("myoElbowPose1D6MRandom-v0")
obs, info = env.reset()
pi = PPO("MlpPolicy", env, verbose=0)
pi.learn(total_timesteps=1000)  # smoke test; use ≥200_000 for task-specific synergies


In [ ]:
base = env.unwrapped
data_store = []
obs, info = env.reset()
for _ in range(10):
    obs, info = env.reset()
    for _ in range(100):
        a, _ = pi.predict(obs, deterministic=True)
        obs, r, terminated, truncated, ifo = env.step(a)
        data_store.append(
            {
                "action": np.asarray(a).copy(),
                "jpos": base.data.qpos.copy(),
                "mlen": base.data.actuator_length.copy(),
                "act": base.data.act.copy(),
            }
        )
        if terminated or truncated:
            obs, info = env.reset()

env.close()
print("Collected", len(data_store), "samples")


In [ ]:
def VAF(W, H, A):
    """
    Args:
        W: ndarray, m x rank matrix, m-muscles x activation coefficients obtained from (# rank) nmf
        H: ndarray, rank x L matrix, basis vectors obtained from nmf where L is the length of the signal
        A: ndarray, m x L matrix, original time-invariant sEMG signal
    Returns:
        global_VAF: float, VAF calculated for the entire A based on the W&H
        local_VAF: 1D array, VAF calculated for each muscle (column) in A based on W&H
    """
    SSE_matrix = (A - np.dot(W, H))**2
    SST_matrix = (A)**2

    global_SSE = np.sum(SSE_matrix)
    global_SST = np.sum(SST_matrix)
    global_VAF = 100 * (1 - global_SSE / global_SST)

    local_SSE = np.sum(SSE_matrix, axis = 0)
    local_SST = np.sum(SST_matrix, axis = 0)
    local_VAF = 100 * (1 - np.divide(local_SSE, local_SST))

    return global_VAF, local_VAF

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import NMF

# NMF needs (n_samples, n_muscles) with non-negative entries.
act = np.clip(np.array([dd["act"] for dd in data_store]), 0.0, None)
n_muscles = act.shape[1]
sample_points = [k for k in range(1, n_muscles + 1)]

VAFstore = []
for isyn in sample_points:
    nmf_model = NMF(n_components=isyn, init="random", random_state=0, max_iter=400)
    W = nmf_model.fit_transform(act)
    H = nmf_model.components_
    global_VAF, local_VAF = VAF(W, H, act)
    VAFstore.append(global_VAF)

plt.plot(sample_points, VAFstore, "-o")
plt.xlabel("Number of Muscle Synergies")
plt.ylabel("Explained Variance R^2")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.show()
